# NRAT — добор ТОЛЬКО декабря 2016

Проходит каждый день декабря 2016, сверяет документы с тем, что уже лежит на Google Drive, и докачивает только недостающее. Уже скачанное пропускает.

## Как запускать (по порядку)
1. **Ячейка 1** — установка и подключение Google Drive (разреши доступ).
2. **Ячейка 2** — настройки и функции.
3. **Ячейка 3** — добор декабря. В конце будет заметка, чего не хватало, и файл-отчёт на Drive.

⚠️ Паузы между запросами не уменьшать — иначе бан на 1–2 дня.

In [ ]:
# ==============================================
# ЯЧЕЙКА 1 — установка + подключение Google Drive
# ==============================================
!pip install requests beautifulsoup4 -q

import requests, os, time, re
from bs4 import BeautifulSoup
from urllib.parse import urlencode

from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive подключён")

In [ ]:
# ==============================================
# ЯЧЕЙКА 2 — настройки + функции
# ==============================================
YEAR = 2016
DOWNLOAD_FOLDER = "/content/drive/MyDrive/nrat_pdfs"

BASE_SEARCH_URL = "https://nrat.ukrintei.ua/searchdb/?"
BASE_PARAMS = {
    '_token': '',
    'typeSearch2': 'ok',
    'typeCategory[]': '0',
    'lcSource': '',
    'authorSearch': '',
    'specialnistSearch[]': '0',
    'temaSearch2': '',
    'textSearch': '',
    'registrationNumberSearch': '',
    'firm_id': '0',
    'sortOrder': 'registration_date',
    'sortDir': 'desc',
    'tab': 'big',
}

# --- ПАУЗЫ (НЕ УМЕНЬШАТЬ!) ---
DELAY_BETWEEN_PAGES = 3
DELAY_BETWEEN_FILES = 1
DELAY_BETWEEN_DAYS  = 5

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9,uk;q=0.8',
})

def day_folder_for(year, date_from):
    return os.path.join(DOWNLOAD_FOLDER, str(year), date_from[:7], date_from)

def refresh_token():
    try:
        r = session.get("https://nrat.ukrintei.ua/searchdb/", timeout=30)
        tok = BeautifulSoup(r.content, 'html.parser').find('input', attrs={'name': '_token'})
        if tok and tok.get('value'):
            BASE_PARAMS['_token'] = tok['value']
            print("  🔑 _token обновлён")
            return True
    except Exception as e:
        print(f"  ⚠️ не удалось обновить _token ({e})")
    return False

def build_search_url(date_from, date_to, page=1):
    params = BASE_PARAMS.copy()
    params['dateFromSearch'] = date_from
    params['dateToSearch'] = date_to
    params['pa'] = str(page)
    return BASE_SEARCH_URL + urlencode(params, doseq=True)

def extract_total_results(soup):
    try:
        m = re.search(r'Знайдено документів:\s*(\d+)', soup.get_text(" ", strip=True))
        if m:
            return int(m.group(1))
    except Exception:
        pass
    return 0

def get_search_results_page(url, retry_count=3):
    for attempt in range(retry_count):
        try:
            response = session.get(url, timeout=60)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            results = []
            for card in soup.find_all('div', class_='my-card-body'):
                link_tag = card.find('a', target='_blank')
                if link_tag and link_tag.get('href'):
                    reg_number = link_tag.get_text(strip=True)
                    detail_url = link_tag['href']
                    doc_id = detail_url.rstrip('/').split('/')[-1]
                    pdf_url = f"https://dir.ukrintei.ua/view/ok/{doc_id}"
                    results.append({'registration': reg_number, 'pdf_url': pdf_url, 'doc_id': doc_id})
            return results, soup
        except Exception as e:
            print(f"    ⚠️ загрузка выдачи, попытка {attempt + 1}/{retry_count}: {e}")
            if attempt < retry_count - 1:
                time.sleep(5)
    return None, None

def download_pdf(pdf_url, registration, doc_id, folder, retry_count=2):
    safe_reg = re.sub(r'[^\w\-_.]', '_', registration)
    filename = re.sub(r'_+', '_', f"{safe_reg}_{doc_id}.pdf")
    filepath = os.path.join(folder, filename)
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        return 'skip'
    for attempt in range(retry_count):
        try:
            response = session.get(pdf_url, stream=True, timeout=60)
            if response.status_code == 404:
                return 'notpdf'
            response.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            if os.path.getsize(filepath) > 0:
                with open(filepath, 'rb') as f:
                    if f.read(4).startswith(b'%PDF'):
                        return 'ok'
                os.remove(filepath)
                return 'notpdf'
            os.remove(filepath)
            return 'empty'
        except Exception as e:
            print(f"    ❌ ошибка скачивания: {e}")
            if attempt < retry_count - 1:
                time.sleep(3)
    return 'fail'

def scrape_day(date_from, date_to, day_folder):
    os.makedirs(day_folder, exist_ok=True)
    page = 1
    page_size = None
    total_pages = None
    seen_ids = set()
    stats = {'ok': 0, 'skip': 0, 'fail': 0, 'notpdf': 0}
    day_ok = True
    while True:
        url = build_search_url(date_from, date_to, page)
        results, soup = get_search_results_page(url)
        if results is None:
            print("    ⚠️ страница выдачи не загрузилась — день будет повторён")
            day_ok = False
            break
        if page == 1:
            total_results = extract_total_results(soup)
            page_size = len(results) if results else 10
            total_pages = ((total_results + 9) // 10) if total_results else None
            if total_results:
                print(f"    найдено {total_results} рез., {total_pages} стр.")
        if not results:
            if page == 1:
                print("    нет результатов")
            break
        new_results = [r for r in results if r['doc_id'] not in seen_ids]
        if page > 1 and not new_results:
            break
        for r in new_results:
            seen_ids.add(r['doc_id'])
        for i, r in enumerate(new_results, 1):
            status = download_pdf(r['pdf_url'], r['registration'], r['doc_id'], day_folder)
            stats[status if status in stats else 'fail'] += 1
            # пауза только если реально качали файл; уже скачанные ('skip') запрос не делают
            if status != 'skip' and i < len(new_results):
                time.sleep(DELAY_BETWEEN_FILES)
        if total_pages is not None:
            go_next = page < total_pages
        else:
            go_next = bool(page_size and len(results) >= page_size)
        if not go_next or page >= 85:
            break
        page += 1
        time.sleep(DELAY_BETWEEN_PAGES)
    return stats, day_ok

print("✅ Настройки и функции загружены")

In [ ]:
# ==============================================
# ЯЧЕЙКА 3 — ДОБОР ДЕКАБРЯ 2016
# ==============================================
refresh_token()

days = [f"{YEAR}-12-{d:02d}" for d in range(1, 32)]
grand = {'ok': 0, 'skip': 0, 'fail': 0, 'notpdf': 0}
added_days = []
net_fail_days = []

print(f"🔁 Добор декабря {YEAR}: {len(days)} дней\n" + "=" * 60)
for idx, d in enumerate(days, 1):
    stats, day_ok = scrape_day(d, d, day_folder_for(YEAR, d))
    for k in grand:
        grand[k] += stats[k]
    note = ""
    if stats['ok'] > 0:
        added_days.append((d, stats['ok']))
        note = f"   ⬇️ ДОГРУЖЕНО {stats['ok']}"
    if not day_ok:
        net_fail_days.append(d)
        note += "   ⚠️ страница не догрузилась — запусти ячейку ещё раз"
    print(f"{d} ({idx}/{len(days)}): новых {stats['ok']}, уже было {stats['skip']}, без PDF {stats['notpdf']}{note}")
    time.sleep(DELAY_BETWEEN_DAYS)

print("\n" + "=" * 60)
print(f"ИТОГ декабрь {YEAR}: догружено {grand['ok']}, уже было {grand['skip']}, без файла {grand['notpdf']}")
if added_days:
    print(f"\n📌 Не хватало в {len(added_days)} днях — догружено:")
    for d, n in added_days:
        print(f"     {d}: +{n}")
else:
    print("\n✅ Пропусков не найдено — весь декабрь уже на Drive.")
if net_fail_days:
    print(f"\n⚠️ Дни с ошибкой сети (запусти ячейку ещё раз): {net_fail_days}")

report_path = os.path.join(DOWNLOAD_FOLDER, f"_report_december_{YEAR}.txt")
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(f"ДОБОР ДЕКАБРЯ {YEAR}\n")
    f.write(f"Догружено: {grand['ok']}; уже было: {grand['skip']}; без файла: {grand['notpdf']}\n\n")
    if added_days:
        f.write("Не хватало (догружено):\n")
        for d, n in added_days:
            f.write(f"  {d}: +{n}\n")
    else:
        f.write("Пропусков не найдено.\n")
    if net_fail_days:
        f.write(f"\nДни с ошибкой сети (повторить): {net_fail_days}\n")
print(f"\n📝 Заметка сохранена: {report_path}")